# 07 — Supplementary material

Builds `supplementary.tex` (and compiles `supplementary.pdf` when `pdflatex` is available) in the folder of
the journal template, from the outputs of notebooks 03, 03b, 04 and 05: Table S1 (algorithms), Table S2
(feature generators with automatic selection), Table S3 (regret of the plant/do-not-plant decision),
Table S4 (municipalities whose best sowing date is 15 March), Figure S1 (costs and prices) and Figure S2
(reliability of the probability of loss).

In [1]:
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
RESULTADOS, FIGURAS, DADOS = RAIZ / "resultados", RAIZ / "figuras", RAIZ / "dados"
DESTINO = RAIZ / "Wiley_LaTeX_Manuscript_Template_for_the_Journal_of_Agronomy_and_Crop_Science"
(DESTINO / "figures").mkdir(exist_ok=True)
for f in ["fig_custos_precos.png", "fig_confiabilidade_prejuizo.png"]:
    shutil.copy(FIGURAS / f, DESTINO / "figures" / f)


def num(x, d=0):
    s = f"{x:,.{d}f}"
    return s.replace("-", "$-$")

In [2]:
# Table S1 — algorithms (notebook 03b)
alg = pd.read_csv(RESULTADOS / "comparacao_algoritmos.csv", index_col=0)
NOME = {"HGB_shallow_monotonic (E)": "HGB, shallow, monotonic (model E)", "HGB_shallow (D)": "HGB, shallow (model D)"}
linhas = []
for nome, r in alg.iterrows():
    ic = "--" if nome.startswith("HGB_shallow_monotonic") else f"{num(r.dRMSE_vs_E)} ({num(r.CI95_low)}; {num(r.CI95_high)})"
    linhas.append(f"{NOME.get(nome, nome).replace('_', ' ')} & {r.R2:.3f} & {r.R2_annual:.3f} & {num(r.RMSE_kg_ha)} & {ic} \\\\")
tab_s1 = "\n".join(linhas)

# Table S2 — automatic selection (notebook 03)
sel = pd.read_csv(RESULTADOS / "selecao_automatica_geradores.csv", index_col=0)
linhas = []
for nome, r in sel.iterrows():
    g, k = nome.rsplit("_k", 1)
    ic = "--" if g == "MZA" else f"{num(r.dif_RMSE_vs_MZA)} ({num(r.IC95_inf)}; {num(r.IC95_sup)})"
    linhas.append(f"{g} & {k} & {r.R2:.3f} & {r.R2_anual_estadual:.3f} & {num(r.RMSE_kg_ha)} & {ic} \\\\")
tab_s2 = "\n".join(linhas)

# Table S3 — regret (notebook 05)
arr = pd.read_csv(RESULTADOS / "arrependimento_decisao.csv", index_col=0)
ROT = {"always sow": "Always sow", "perfect information": "Perfect information",
       "A_media_historica": "A Historical mean", "B_MZA_calibrado": "B Calibrated MZA", "C_ML_puro": "C Pure ML",
       "D_hibrido": "D MZA hybrid", "E_hibrido_monotonico": "E MZA hybrid, monotonic", "F_hibrido_apsim": "F APSIM hybrid",
       "H_hibrido_aquacrop": "H AquaCrop hybrid", "I_hibrido_dssat": "I DSSAT hybrid", "J_hibrido_todos": "J All models",
       "K_hibrido_svr": "K MZA hybrid, SVR", "L_ensemble_E_SVR": "L Ensemble E + K"}
linhas = []
for nome, r in arr.iterrows():
    rot = ROT[nome]
    vs = r.get("vs_E_IC95")
    vs = "--" if (not isinstance(vs, str) or nome.startswith("E_")) else vs.replace("-", "$-$").replace("+", "$+$")
    linhas.append(f"{rot} & {num(r['regret_R$_ha'])} & {r['sows_%']:.0f} & {r['correct_decision_%']:.0f} & {num(r['realized_margin_R$_ha'])} & {vs} \\\\")
tab_s3 = "\n".join(linhas)
print(tab_s1[:300]); print(tab_s3[:300])

Ensemble HGB nested + SVR nested & 0.437 & 0.681 & 1,052 & $-$39 ($-$75; $-$7) \\
SVR nested & 0.436 & 0.718 & 1,053 & $-$38 ($-$91; 6) \\
Ensemble SVR nested + E & 0.435 & 0.688 & 1,054 & $-$37 ($-$63; $-$16) \\
SVR fixed & 0.415 & 0.708 & 1,072 & $-$20 ($-$78; 32) \\
HGB monotonic nested & 0.405 &
Always sow & 448 & 100 & 45 & $-$4 & $+$118 ($-$66; $+$304) \\
A Historical mean & 307 & 57 & 60 & 137 & $-$25 ($-$105; $+$60) \\
B Calibrated MZA & 313 & 56 & 60 & 131 & $-$19 ($-$40; $+$4) \\
C Pure ML & 320 & 58 & 59 & 124 & $-$12 ($-$47; $+$21) \\
D MZA hybrid & 325 & 57 & 59 & 119 & $-$9 ($-$30


In [3]:
# Table S4 — municipalities whose best date is 15 March: flowering heat for sowing on 5 January × 15 March
dec = pd.read_csv(RESULTADOS / "melhor_data_semeadura_pr.csv", index_col=0)
mun = pd.read_csv(DADOS / "municipios_pr.csv").set_index("cod_ibge")
g15 = dec.index[dec["melhor_data_zarc"] == "15/03"]
sim = pd.read_parquet(RESULTADOS / "simulacao_mza_pr.parquet",
                      columns=["cod_ibge", "semeadura", "Tmax_floracao", "calor_floracao"])
sim = sim[sim["cod_ibge"].isin(g15) & sim["semeadura"].isin(["05/01", "15/03"])]
calor = sim.groupby(["cod_ibge", "semeadura"])[["Tmax_floracao", "calor_floracao"]].mean().unstack()
prod = pd.read_parquet(RESULTADOS / "produtividade_prevista_pr.parquet", columns=["cod_ibge", "semeadura", "rend_P50_kg_ha"])
p50 = prod[prod["cod_ibge"].isin(g15)].groupby(["cod_ibge", "semeadura"])["rend_P50_kg_ha"].mean().unstack()
t4 = pd.DataFrame({"municipio": mun.loc[g15, "municipio"], "meso": mun.loc[g15, "mesorregiao"],
                   "alt": mun.loc[g15, "altitude_m"],
                   "tmax05": calor[("Tmax_floracao", "05/01")], "tmax15": calor[("Tmax_floracao", "15/03")],
                   "calor05": calor[("calor_floracao", "05/01")], "calor15": calor[("calor_floracao", "15/03")],
                   "ganho": 100 * (p50["15/03"] / p50["05/01"] - 1)}).sort_values(["meso", "municipio"])
tab_s4 = "\n".join(f"{r.municipio} & {r.meso.replace(' Paranaense', '')} & {r.alt:.0f} & {r.tmax05:.1f} & {r.tmax15:.1f} & "
                    f"{r.calor05:.1f} & {r.calor15:.1f} & {num(r.ganho, 1)} \\\\" for r in t4.itertuples())
print(len(t4), "municipalities;", t4[["tmax05", "tmax15", "calor05", "calor15", "ganho"]].mean().round(2).to_dict())

27 municipalities; {'tmax05': 32.31, 'tmax15': 27.2, 'calor05': 1.9, 'calor15': 0.18, 'ganho': 8.9}


In [4]:
doc = r"""\documentclass[10pt,a4paper]{article}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{times}
\usepackage[a4paper,margin=2cm]{geometry}
\usepackage{booktabs,longtable,graphicx}
\usepackage[hidelinks]{hyperref}
\renewcommand{\thetable}{S\arabic{table}}
\renewcommand{\thefigure}{S\arabic{figure}}
\setlength{\parindent}{0pt}
\begin{document}
{\large\bfseries Supplementary material}\\[4pt]
Hybrid crop modelling and machine learning for second-season maize yield and the economic risk of sowing
dates in Paraná, Brazil\\[2pt]
\textit{Journal of Agronomy and Crop Science}

\begin{table}[!ht]
\centering
\caption{Leave-one-year-out comparison of machine learning algorithms trained on the MZA-FAO features
(6,806 municipality-years). Default, library defaults; shallow, shallow and regularized; nested,
hyperparameters chosen within the training years of each fold. $\Delta RMSE$, median difference relative
to model E (kg ha$^{-1}$) with 95\% bootstrap interval over years.}
\small
\begin{tabular}{lrrrl}
\toprule
Algorithm & $R^2$ & $R^2$ annual & $RMSE$ (kg ha$^{-1}$) & $\Delta RMSE$ vs E \\
\midrule
TAB_S1
\bottomrule
\end{tabular}
\end{table}

\begin{table}[!ht]
\centering
\caption{Comparison of the feature generators with an identical automatic selection inside each fold
(the $k$ candidates with the highest absolute Spearman correlation with the training anomaly, skipping
candidates correlated above 0.9 with one already chosen) and the same unconstrained gradient boosting.
$\Delta RMSE$ relative to MZA-FAO with the same $k$ (kg ha$^{-1}$), with 95\% bootstrap interval.}
\small
\begin{tabular}{lrrrrl}
\toprule
Generator & $k$ & $R^2$ & $R^2$ annual & $RMSE$ (kg ha$^{-1}$) & $\Delta RMSE$ vs MZA-FAO \\
\midrule
TAB_S2
\bottomrule
\end{tabular}
\end{table}

\begin{table}[!ht]
\centering
\caption{Regret of the plant/do-not-plant decision for each yield model, with leave-one-year-out
predictions and observed yields, prices and operating costs (R\$ ha$^{-1}$). Difference relative to model
E with 95\% bootstrap interval over years.}
\small
\begin{tabular}{lrrrrl}
\toprule
Strategy / model & Regret & Sows (\%) & Correct (\%) & Realized margin & Regret vs E \\
\midrule
TAB_S3
\bottomrule
\end{tabular}
\end{table}

\clearpage
{\footnotesize\setlength{\tabcolsep}{4pt}
\begin{longtable}{llrrrrrr}
\caption{Municipalities whose best sowing date (certainty equivalent, $r_r$ = 2, ZARC-suitable dates) was
15 March: altitude, mean maximum temperature at flowering ($T_{max}$, \textdegree{}C) and days with
$T_{max} \ge$ 35 \textdegree{}C at flowering for sowing on 5 January and 15 March (1999--2025), and change
in the median predicted yield from 5 January to 15 March.}\\
\toprule
Municipality & Mesoregion & Alt. (m) & \multicolumn{2}{c}{$T_{max}$ (\textdegree{}C)} & \multicolumn{2}{c}{Heat days} & Yield change \\
 & & & 5 Jan & 15 Mar & 5 Jan & 15 Mar & (\%) \\
\midrule
\endfirsthead
\toprule
Municipality & Mesoregion & Alt. (m) & \multicolumn{2}{c}{$T_{max}$ (\textdegree{}C)} & \multicolumn{2}{c}{Heat days} & Yield change \\
 & & & 5 Jan & 15 Mar & 5 Jan & 15 Mar & (\%) \\
\midrule
\endhead
TAB_S4
\bottomrule
\end{longtable}}

\begin{figure}[!ht]
\centering
\includegraphics[width=\textwidth]{figures/fig_custos_precos.png}
\caption{Real production costs of second-season maize in Paraná by harvest year (total, operating and
variable cost; R\$ ha$^{-1}$ of December 2025) and annual mean real price received by farmers, with the
monthly seasonal price index (inset).}
\end{figure}

\begin{figure}[!ht]
\centering
\includegraphics[width=0.62\textwidth]{figures/fig_confiabilidade_prejuizo.png}
\caption{Reliability of the out-of-sample probability of loss (municipal yield below the break-even yield
over operating cost) for selected yield models, with the Brier score of each model.}
\end{figure}

\end{document}
"""
doc = (doc.replace("TAB_S1", tab_s1).replace("TAB_S2", tab_s2).replace("TAB_S3", tab_s3).replace("TAB_S4", tab_s4))
(DESTINO / "supplementary.tex").write_text(doc)
if shutil.which("pdflatex"):
    for _ in range(2):
        r = subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "supplementary.tex"],
                           cwd=DESTINO, capture_output=True, text=True)
    print("pdflatex:", "ok" if r.returncode == 0 else r.stdout[-2000:])
    for ext in ["aux", "log", "out"]:
        (DESTINO / f"supplementary.{ext}").unlink(missing_ok=True)
print((DESTINO / "supplementary.tex").exists(), (DESTINO / "supplementary.pdf").exists())

pdflatex: ok
True True
